# Fernow Experimental Forest — Forest Management Effects on Ecosystem Carbon
## Interactive analysis notebook — written for the ArcGIS Pro Notebook environment

**How to use:** open this notebook inside ArcGIS Pro (Insert ▸ New Notebook, or add this .ipynb to your project),
set `DATA` in the first code cell to the folder containing the `gdb_export` CSVs, then run cells top-to-bottom.
Every statistic is computed live from the raw exported tables — nothing is hard-coded. All code uses only the
default `arcgispro-py3` environment (pandas / numpy / scipy / matplotlib); no cloned environment needed.
`arcpy` is **not required** for any cell (the analysis reads CSVs), so this also runs in plain Jupyter.

**Site / design recap:**
- **WS4 – REF**: unmanaged reference (last logged 1907–1915)
- **WS2 – DLC**: diameter-limit cut (≥43 cm removed), last harvest 2020 (2 yr before sampling)
- **WS5 – STS**: single-tree selection (≥28 cm removed). Written removal records: WS5A (5 of 6 plots) last cut
  winter 2018–19 (~3.5 yr before sampling), WS5B (1 plot) 2013. The thesis/handoff date "2013 for all of WS5" is
  superseded — see notebook 02 §0.
- July 2022 sampling. Trees: 0.1-ac (404.708 m²) USFS permanent growth plots, 6/watershed. Soils: 3 subplots ×
  3 depths (OH, 0–20, 20–40 cm) per plot (162 samples); density fractionation (LF / hPOM / MAOM) on plot
  composites for the two mineral depths (36 samples).

**Statistical conventions (used by every section):**
- Unit of replication = **plot** (n = 6/watershed/depth). Subplots are averaged first — subplot-level n = 18
  inflates significance (pseudoreplication; Hurlbert 1984, 2004; Oksanen 2001) and is never used for inference.
- One watershed per treatment → inference is **conditional on these three watersheds**; treatments are not
  replicated at the watershed scale. Aboveground vs. belowground comparisons are watershed-level by design
  (soil plots are linked to subcompartment, not to individual tree plots — source geodatabase QC log).
- Primary test: one-way ANOVA + Dunnett's (managed vs. REF). Levene's + Shapiro–Wilk are checked on every model;
  Welch's ANOVA and Kruskal–Wallis are always reported alongside and lead the interpretation when assumptions fail.
- Benjamini–Hochberg FDR across the exploratory soil battery. Non-significant results = **"not detected"**
  (power is reported), never "no difference."
- Light-fraction (LF) carbon is the **primary** fraction metric (project decision, Sept 2026); the dry-mass
  version is supplementary. Three left-censored LF non-detects (WS5, 20–40 cm) are handled with an explicit
  substitution sensitivity table, never silently dropped or zero-filled.


In [ ]:
# ============================== CONFIG — edit this cell only ==============================
# Point DATA at the folder that contains the gdb_export CSVs (trees.csv, soil_samples.csv, soil_fractions.csv).
# ArcGIS Pro example:  DATA = r'C:\Users\you\Documents\Fernow\gdb_export'
import os
DATA = r'gdb_export'          # <-- EDIT ME if the notebook is not sitting next to the gdb_export folder

# If True, the ecosystem-C figure scales POM/MAOM so their sum equals bulk soil C (mass-balance-consistent stack).
# If False, POM/MAOM are plotted exactly as measured (fraction recovery 86-113% of bulk; both are defensible,
# the choice just has to be stated in the caption).
SCALE_FRACTIONS_TO_BULK = True
# ==========================================================================================

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.width', 140); pd.set_option('display.max_columns', 40)

ORDER  = ['WS4', 'WS2', 'WS5']                       # REF, DLC, STS — keep this order everywhere
LABELS = {'WS4': 'REF', 'WS2': 'DLC', 'WS5': 'STS'}
COLORS = {'WS4': '#4C6444', 'WS2': '#B0733A', 'WS5': '#3A6EA5'}

for f in ['trees.csv', 'soil_samples.csv', 'soil_fractions.csv']:
    assert os.path.exists(os.path.join(DATA, f)), f'Cannot find {f} in DATA — fix the DATA path above.'
print('scipy', __import__('scipy').__version__, '| pandas', pd.__version__, '| data folder OK')


In [ ]:
# ---------- Shared statistical helpers (no packages beyond scipy needed) ----------

def welch_anova(groups):
    """Welch's heteroscedasticity-robust one-way ANOVA (matches R oneway.test var.equal=FALSE)."""
    k = len(groups); n = np.array([len(g) for g in groups], float)
    m = np.array([g.mean() for g in groups]); v = np.array([g.var(ddof=1) for g in groups])
    w = n / v
    grand = np.sum(w * m) / np.sum(w)
    num = np.sum(w * (m - grand) ** 2) / (k - 1)
    tmp = np.sum((1 - w / np.sum(w)) ** 2 / (n - 1))
    F = num / (1 + 2 * (k - 2) / (k ** 2 - 1) * tmp)
    df2 = (k ** 2 - 1) / (3 * tmp)
    return F, 1 - stats.f.cdf(F, k - 1, df2)

def dunnett_mc(groups, n_sim=400_000, seed=12345):
    """Dunnett's many-to-one test (two-sided) vs groups[0], by Monte Carlo integration of the
    multivariate t distribution. Identical in definition to scipy.stats.dunnett (SciPy >= 1.11);
    Monte Carlo error on p is ~0.0005 at p = 0.05 with the default n_sim. Works on any SciPy version."""
    groups = [np.asarray(g, float) for g in groups]
    n = np.array([len(g) for g in groups], float)
    k = len(groups); df = int(n.sum() - k)
    sp2 = sum(((g - g.mean()) ** 2).sum() for g in groups) / df
    t_obs = np.array([(g.mean() - groups[0].mean()) / np.sqrt(sp2 * (1 / len(g) + 1 / n[0])) for g in groups[1:]])
    lam = np.sqrt(n[1:] / (n[1:] + n[0]))
    R = np.outer(lam, lam); np.fill_diagonal(R, 1.0)
    rng = np.random.default_rng(seed)
    Z = rng.multivariate_normal(np.zeros(k - 1), R, size=n_sim)
    V = np.sqrt(rng.chisquare(df, size=n_sim) / df)
    tmax = np.abs(Z / V[:, None]).max(axis=1)
    return [float((tmax >= abs(t)).mean()) for t in t_obs]


def dunnett_vs_control(groups):
    """Dunnett vs groups[0]: SciPy's exact version when available (>= 1.11), otherwise the
    Monte Carlo implementation above (same test, not a substitute test)."""
    try:
        from scipy.stats import dunnett as _d
        return list(_d(*groups[1:], control=groups[0]).pvalue), 'Dunnett'
    except ImportError:
        return dunnett_mc(groups), 'Dunnett (Monte Carlo)'


def bh_fdr(pvals):
    """Benjamini-Hochberg adjusted p-values (manual — statsmodels not in arcgispro-py3)."""
    p = np.asarray(pvals, float); m = len(p)
    order = np.argsort(p); ranked = p[order] * m / (np.arange(m) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    out = np.empty(m); out[order] = np.minimum(ranked, 1.0)
    return out

def anova_power(groups, alpha=0.05):
    """Post-hoc power for the one-way ANOVA via the noncentral F distribution."""
    m = [g.mean() for g in groups]
    pooled = np.mean([g.var(ddof=1) for g in groups])
    f_eff = np.sqrt(np.mean([(x - np.mean(m)) ** 2 for x in m]) / pooled)   # Cohen's f
    N = sum(len(g) for g in groups); k = len(groups)
    nc = f_eff ** 2 * N
    fcrit = stats.f.ppf(1 - alpha, k - 1, N - k)
    return f_eff, stats.ncf.sf(fcrit, k - 1, N - k, nc)

def battery(groups):
    """Full assumption-checked battery. groups[0] must be REF."""
    groups = [np.asarray(g, float) for g in groups]
    groups = [g[~np.isnan(g)] for g in groups]
    F, p = stats.f_oneway(*groups)
    _, lev_p = stats.levene(*groups)
    _, sh_p = stats.shapiro(np.concatenate([g - g.mean() for g in groups]))
    H, pk = stats.kruskal(*groups)
    Fw, pw = welch_anova(groups)
    dun_p, dun_method = dunnett_vs_control(groups)
    f_eff, power = anova_power(groups)
    return dict(n=[len(g) for g in groups], means=[g.mean() for g in groups],
                sds=[g.std(ddof=1) for g in groups], F=F, p=p, lev_p=lev_p, sh_p=sh_p,
                Fw=Fw, pw=pw, H=H, pk=pk, dunnett_p=dun_p, dunnett_method=dun_method,
                f_eff=f_eff, power=power)

print('Helpers loaded.')


## Section 1 — Aboveground carbon (AGB-C)

`trees.csv`: per-stem inventory, WS2/WS4/WS5 only, pre-cleaned per the Phase-0 QC log (partial plot `WS4-A-11`
excluded, DBH converted inches→cm, mis-allometried legacy `agb` column discarded). Biomass = Jenkins et al. (2003)
species-group allometry (`agb_kg_jenkins`); C = 0.48 × biomass. Plot area = 404.708 m² (0.1 ac), confirmed from the
plot-footprint geometry itself (`plots_footprint.csv` Shape_Area), which supersedes the earlier FIA-24-ft
placeholder from before the true footprint was located.


In [ ]:
trees = pd.read_csv(os.path.join(DATA, 'trees.csv'))
trees = trees[trees.dbh_cm.notna()].copy()
PLOT_AREA_M2 = 404.70782
trees['agbC_kg'] = trees['agb_kg_jenkins'] * 0.48

agb_plot = trees.groupby(['watershed', 'plot_id'])['agbC_kg'].sum().reset_index()
agb_plot['MgC_ha'] = agb_plot['agbC_kg'] / 1000 / (PLOT_AREA_M2 / 10000)

summary = agb_plot.groupby('watershed')['MgC_ha'].agg(['count', 'mean', 'std']).reindex(ORDER)
summary.index = [LABELS[w] for w in summary.index]
print('AGB-C by watershed (Mg C/ha), plot level:'); print(summary.round(1))

groups = [agb_plot[agb_plot.watershed == w]['MgC_ha'].values for w in ORDER]
r = battery(groups)
print(f"\nANOVA F={r['F']:.2f} p={r['p']:.4f} | Levene p={r['lev_p']:.4f} (fails -> lead with Welch/KW) | Shapiro p={r['sh_p']:.4f}")
print(f"Welch p={r['pw']:.4f} | Kruskal-Wallis p={r['pk']:.4f}")
print(f"{r['dunnett_method']} vs REF: DLC p={r['dunnett_p'][0]:.4f}, STS p={r['dunnett_p'][1]:.4f}")
print(f"Cohen's f={r['f_eff']:.2f}, power={r['power']:.2f}")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for i, w in enumerate(ORDER):
    vals = agb_plot[agb_plot.watershed == w]['MgC_ha'].values
    ax.bar(i, vals.mean(), yerr=vals.std(ddof=1), capsize=5, color=COLORS[w], alpha=0.85)
    ax.scatter(np.full(len(vals), i) + np.random.uniform(-0.12, 0.12, len(vals)), vals,
               color='k', zorder=3, s=18)
ax.set_xticks(range(3)); ax.set_xticklabels([LABELS[w] for w in ORDER])
ax.set_ylabel('Aboveground C (Mg ha$^{-1}$)')
ax.set_title('Fig. 1  AGB-C by watershed (mean ± SD; points = plots)')
plt.tight_layout(); plt.show()


**Where the difference lives — size structure.** The REF surplus should sit almost entirely in large legacy trees
(≥50 cm DBH), with the mid-size cohort (≥27.9 cm) invariant to management — check both:


In [ ]:
full_plots = agb_plot[['watershed', 'plot_id']].drop_duplicates()
mid = trees[trees.dbh_cm >= 27.9].groupby(['watershed', 'plot_id']).size().reset_index(name='n')
mid = full_plots.merge(mid, how='left').fillna(0)
g = [mid[mid.watershed == w]['n'].values for w in ORDER]
print(f'Stems >=27.9 cm per plot: ANOVA p={stats.f_oneway(*g)[1]:.3f}  (expected n.s.)')

big = trees[trees.dbh_cm >= 50]
print('\nStems >=50 cm by watershed:'); print(big.groupby('watershed').size().reindex(ORDER).fillna(0).astype(int))
ref_t = trees[trees.watershed == 'WS4']
share = ref_t.loc[ref_t.dbh_cm >= 50, 'agbC_kg'].sum() / ref_t['agbC_kg'].sum() * 100
print(f'\nShare of pooled REF AGB-C in >=50 cm stems: {share:.0f}%')
print('Largest stem (cm):', trees.groupby('watershed')['dbh_cm'].max().reindex(ORDER).round(1).to_dict())


## Section 2 — Bulk soil C, N, C:N, bulk density, fine roots (plot level, n = 6)

`soil_samples.csv` (162 rows = 3 × 6 × 3 depths × 3 subplots). Subplots are averaged to the plot before any test.
BD is one core per plot × depth in the source (applied to subplots), so the plot mean is the measured value, not a
pseudo-average.


In [ ]:
soil = pd.read_csv(os.path.join(DATA, 'soil_samples.csv'))
resp = ['soil_C_gm2', 'soil_N_gm2', 'soil_CN', 'fine_root_gm2', 'BD_gcm3']
soil_plot = soil.groupby(['watershed', 'soil_plot', 'depth_code'])[resp].mean().reset_index()

rows = []
for var in resp:
    for depth in ['OH', '0-20', '20-40']:
        if var == 'BD_gcm3' and depth == 'OH':
            continue
        d = soil_plot[soil_plot.depth_code == depth]
        g = [d[d.watershed == w][var].dropna().values for w in ORDER]
        r = battery(g)
        rows.append(dict(variable=var, depth=depth,
                         REF=round(r['means'][0], 2), DLC=round(r['means'][1], 2), STS=round(r['means'][2], 2),
                         ANOVA_p=round(r['p'], 4), Welch_p=round(r['pw'], 4), KW_p=round(r['pk'], 4),
                         Levene_p=round(r['lev_p'], 4), Shapiro_p=round(r['sh_p'], 4),
                         Dunnett_DLC=round(r['dunnett_p'][0], 4), Dunnett_STS=round(r['dunnett_p'][1], 4),
                         f=round(r['f_eff'], 2), power=round(r['power'], 2)))
soil_results = pd.DataFrame(rows)

# FDR across the exploratory battery (manual Benjamini-Hochberg)
soil_results['ANOVA_p_FDR'] = bh_fdr(soil_results['ANOVA_p']).round(4)
soil_results['KW_p_FDR'] = bh_fdr(soil_results['KW_p']).round(4)
soil_results


**Expected reading:** nothing in the bulk-soil battery survives FDR at α = 0.05 at plot-level n = 6 — the
organic-horizon C/N losses under management are directionally consistent **trends** (KW ≈ 0.02 uncorrected),
power is ≤ ~0.5 throughout, and every non-significant contrast is reported as *not detected*, not *no difference*.
This matches the pseudoreplication-corrected reanalysis already adopted for the manuscript and Gilliam et al.
(2004), who also found few soil C/N differences under these treatments at Fernow.


In [ ]:
# Total mineral-soil C stock, 0-40 cm (Mg/ha) — the belowground headline stock number
wide = soil_plot.pivot_table(index=['watershed', 'soil_plot'], columns='depth_code',
                             values='soil_C_gm2').reset_index()
wide['SoilC_0_40'] = (wide['0-20'] + wide['20-40']) / 100
g = [wide[wide.watershed == w]['SoilC_0_40'].values for w in ORDER]
r = battery(g)
print('Soil C 0-40 cm (Mg/ha):', {LABELS[w]: round(m, 1) for w, m in zip(ORDER, r['means'])})
print(f"ANOVA p={r['p']:.3f} | Welch p={r['pw']:.3f} | KW p={r['pk']:.3f} | power={r['power']:.2f} -> 'not detected'")


## Section 3 — Density fractions: LF, hPOM, MAOM (carbon basis primary)

`soil_fractions.csv`: 36 plot composites (3 × 6 plots × 2 mineral depths; no OH fractionation).
**POM (particulate organic matter) = LF + hPOM** wherever the two are combined below; MAOM is the
mineral-associated pool.

**Censoring:** 3 WS5 20–40 cm composites yielded too little light-fraction material for %C analysis →
`LF_C_gm2` is a left-censored non-detect for those plots (the LF *mass* was still recorded). Handled with a
substitution sensitivity table — the focal DLC-vs-REF contrast never touches the censored group, which is why
it is the manuscript's primary framing at this depth.


In [ ]:
frac = pd.read_csv(os.path.join(DATA, 'soil_fractions.csv'))
sub40 = frac[frac.depth_code == '20-40'].copy()
print(sub40[['watershed', 'soil_plot', 'LF_g', 'LF_C_gm2']].sort_values(['watershed', 'soil_plot']).to_string(index=False))
print('\nLeft-censored LF %C non-detects:', int(sub40['LF_C_gm2'].isna().sum()), '(all WS5)')


In [ ]:
ref = sub40.loc[sub40.watershed == 'WS4', 'LF_C_gm2'].values
dlc = sub40.loc[sub40.watershed == 'WS2', 'LF_C_gm2'].values
sts = sub40.loc[sub40.watershed == 'WS5', 'LF_C_gm2'].values
dl_floor = frac['LF_C_gm2'].dropna().min()     # smallest observed value = detection-limit proxy

sts_cc = sts[~np.isnan(sts)]
F, p = stats.f_oneway(ref, dlc, sts_cc)
t, pt = stats.ttest_ind(dlc, ref, equal_var=False)
print('PRIMARY — subsoil (20-40 cm) light-fraction CARBON:')
print(f'  Means (gC/m2): REF={ref.mean():.0f}  DLC={dlc.mean():.0f}  STS(detected n=3)={sts_cc.mean():.0f}')
print(f'  DLC/REF enrichment: {dlc.mean()/ref.mean():.2f}x')
print(f'  Focal contrast DLC vs REF (Welch t): p={pt:.4f}')
print(f'  3-group ANOVA (complete-case): p={p:.4f}')

print('\nSENSITIVITY — 3 STS non-detects substituted (STS back to n=6):')
for label, val in [('zero', 0.0), ('half detection limit', dl_floor / 2), ('full detection limit', dl_floor)]:
    sts_f = np.where(np.isnan(sts), val, sts)
    print(f'  {label:22s} STS mean={sts_f.mean():7.1f}   ANOVA p={stats.f_oneway(ref, dlc, sts_f)[1]:.4f}')


**Manuscript statement this supports:** two years after diameter-limit cutting, light-fraction carbon at 20–40 cm
is **~2.2× the unmanaged reference** (Welch t p ≈ 0.007; 3-group ANOVA p = 0.004–0.013 across every
censoring-handling choice). Mechanistic framing per the verified literature memo: residue/root-derived
particulate matter incorporated to depth (Nave et al. 2010), a pool the chronosequence literature identifies
as the *vulnerable* deep pool post-harvest (Diochon & Kellman 2009) — i.e., first quantification of this
enrichment under diameter-limit cutting in Appalachian hardwoods, with its fate (decay pulse vs. burial
stabilization) posed as the open question. With the corrected harvest dates, 5 of 6 plots per managed watershed were 2 yr (DLC) and ~3.5 yr (STS) post-harvest,
so DLC vs. STS is primarily an intensity contrast (median 37% vs 25% of AGB-C removed per entry; notebook 02 §3).

Note this **supersedes** the earlier "~2.6×" figure from a prior session — that number is not reproducible from
the authoritative exported data on either the carbon or the mass basis (both give 2.2–2.3×).


In [ ]:
# Supplementary: dry-mass basis (complete data, no censoring) — same story, kept for SI
sub40['LF_massfrac'] = sub40['LF_g'] / sub40['frac_soil_wt_g']
gm = [sub40.loc[sub40.watershed == w, 'LF_massfrac'].values for w in ORDER]
print(f'SI check, LF dry-mass fraction 20-40cm: DLC/REF = {gm[1].mean()/gm[0].mean():.2f}x, '
      f'ANOVA p={stats.f_oneway(*gm)[1]:.4f}, KW p={stats.kruskal(*gm)[1]:.4f}')


In [ ]:
# Full fraction battery, both depths (POM = LF + hPOM computed with half-DL fill for the 3 censored LF values,
# flagged in output; MAOC:totalPOC carried through but pre-flagged as fragile from prior rounds)
frac['LF_C_filled'] = frac['LF_C_gm2'].fillna(dl_floor / 2)
frac['POM_C_gm2'] = frac['LF_C_filled'] + frac['hPOM_C_gm2']

out = []
for var in ['LF_C_gm2', 'hPOM_C_gm2', 'POM_C_gm2', 'MAOM_C_gm2', 'MAOC_tot_POC']:
    for depth in ['0-20', '20-40']:
        d = frac[frac.depth_code == depth]
        g = [d.loc[d.watershed == w, var].dropna().values for w in ORDER]
        r = battery(g)
        out.append(dict(variable=var, depth=depth,
                        REF=round(r['means'][0], 1), DLC=round(r['means'][1], 1), STS=round(r['means'][2], 1),
                        ANOVA_p=round(r['p'], 4), Welch_p=round(r['pw'], 4), KW_p=round(r['pk'], 4),
                        Levene_p=round(r['lev_p'], 4)))
frac_results = pd.DataFrame(out)
frac_results


**Carry-forward flags:** MAOC:total POC (20–40 cm) remains a **trend only** (fails Levene + KW jointly — as
established in earlier project rounds); POM_C rows involving WS5 at 20–40 cm inherit the half-DL fill and are
descriptive there.


## Section 4 — Total ecosystem carbon, with belowground split into POM vs. MAOM

**This is a storage (stock) snapshot for 2022, not a sequestration rate** — the dataset has no verified repeat
measurement, so no accumulation rate is computed (Section 6). Watershed-level by design (n = 1/treatment):
descriptive, no cross-watershed hypothesis test.

Pools stacked, all Mg C/ha:
1. **Aboveground** (Jenkins AGB × 0.48)
2. **Fine roots, 0–40 cm** (biomass × 0.48)
3. **Forest floor (OH) C** (bulk, no fractionation available)
4. **Mineral-soil POM-C (LF + hPOM), 0–40 cm**
5. **Mineral-soil MAOM-C, 0–40 cm**

Fraction recovery is 86–113% of bulk soil C depending on watershed (normal for density fractionation).
`SCALE_FRACTIONS_TO_BULK` (config cell) chooses between (a) scaling POM/MAOM proportionally so their sum equals
measured bulk C — keeps this figure mass-balance-consistent with Section 2 (default) — or (b) plotting fractions
exactly as measured. State whichever you use in the caption.


In [ ]:
# Pool computation (watershed means)
agb_ws = agb_plot.groupby('watershed')['MgC_ha'].mean()

oh_ws = soil_plot[soil_plot.depth_code == 'OH'].groupby('watershed')['soil_C_gm2'].mean() / 100

root_ws = (soil_plot[soil_plot.depth_code.isin(['0-20', '20-40'])]
           .groupby(['watershed', 'soil_plot'])['fine_root_gm2'].sum()
           .groupby('watershed').mean()) / 100 * 0.48

bulk_ws = wide.groupby('watershed')['SoilC_0_40'].mean()

frac_plot = frac.groupby(['watershed', 'soil_plot'])[['POM_C_gm2', 'MAOM_C_gm2']].sum() / 100
frac_ws = frac_plot.groupby('watershed').mean()

recovery = (frac_ws.sum(axis=1) / bulk_ws)
print('Fraction-sum / bulk-C recovery by watershed:', recovery.round(2).to_dict())

if SCALE_FRACTIONS_TO_BULK:
    pom_ws  = frac_ws['POM_C_gm2']  / recovery
    maom_ws = frac_ws['MAOM_C_gm2'] / recovery
    basis_note = 'POM/MAOM scaled to bulk soil C (mass-balance-consistent)'
else:
    pom_ws, maom_ws = frac_ws['POM_C_gm2'], frac_ws['MAOM_C_gm2']
    basis_note = 'POM/MAOM as measured (recovery 86-113% of bulk)'

eco = pd.DataFrame({'Aboveground': agb_ws, 'Fine roots': root_ws, 'Forest floor (OH)': oh_ws,
                    'POM-C (LF+hPOM)': pom_ws, 'MAOM-C': maom_ws}).reindex(ORDER)
eco['Total'] = eco.sum(axis=1)
eco['% belowground'] = ((eco['Total'] - eco['Aboveground']) / eco['Total'] * 100).round(1)
eco['% of mineral soil C as POM'] = (pom_ws / (pom_ws + maom_ws) * 100).round(1)
eco.index = [LABELS[w] for w in eco.index]
print('\nBasis:', basis_note)
eco.round(1)


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
stack_cols = ['Aboveground', 'Fine roots', 'Forest floor (OH)', 'POM-C (LF+hPOM)', 'MAOM-C']
stack_colors = ['#4C6444', '#B08D57', '#8B7355', '#C9A66B', '#5C4A38']
bottom = np.zeros(3)
for col, c in zip(stack_cols, stack_colors):
    ax.bar(eco.index, eco[col], bottom=bottom, label=col, color=c)
    bottom += eco[col].values
for i, tot in enumerate(eco['Total']):
    ax.text(i, tot + 3, f'{tot:.0f}', ha='center', fontsize=9)
ax.set_ylabel('Ecosystem C (Mg ha$^{-1}$)')
ax.set_title('Fig. 3  Ecosystem C storage, 2022 — belowground split into POM vs. MAOM\n(watershed-level, descriptive)')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout(); plt.show()


**Interpretation to carry into Discussion (storage vs. sequestration, stated properly):**
- **Storage:** REF holds the most total C and is the most aboveground-heavy; both managed watersheds hold
  proportionally more of their C belowground, and a larger share of their *mineral-soil* C as POM — the
  faster-cycling, less-protected pool (Sokol & Bradford 2019; Lavallee et al. 2020 for the POM/MAOM framework).
  That compositional shift is a carbon-*quality* result the bulk-C comparison alone would miss, and it is the
  natural bridge to the subsoil-LF enrichment result (Section 3).
- **Sequestration:** no rate is computed — no verified repeat measurement exists in this dataset. Qualitative,
  literature-anchored framing instead: REF's stock is carried by large legacy trees that continue to accumulate C
  (Stephenson et al. 2014; Luyssaert et al. 2008), while DLC and STS are early in post-harvest aboveground
  rebuild from a lower base. Storage ≠ sequestration, and the manuscript says so explicitly rather than letting
  the stock comparison imply a rate claim.


## Section 5 — Consolidated results table (Table 1 draft)

Everything below is regenerated from the objects computed above (no hand-typed numbers).


In [ ]:
def fmt(m, s):
    return f'{m:.1f} ± {s:.1f}'

r_agb = battery([agb_plot[agb_plot.watershed == w]['MgC_ha'].values for w in ORDER])
r_soil = battery([wide[wide.watershed == w]['SoilC_0_40'].values for w in ORDER])
lf_p_range = '0.004–0.013'

table1 = pd.DataFrame([
    ['AGB-C (Mg/ha)', fmt(r_agb['means'][0], r_agb['sds'][0]), fmt(r_agb['means'][1], r_agb['sds'][1]),
     fmt(r_agb['means'][2], r_agb['sds'][2]),
     f"Welch p={r_agb['pw']:.3f}; KW p={r_agb['pk']:.3f}",
     f"Dunnett DLC p={r_agb['dunnett_p'][0]:.3f}, STS p={r_agb['dunnett_p'][1]:.3f}; Levene fails -> Welch/KW lead"],
    ['Mineral soil C 0-40 cm (Mg/ha)', fmt(r_soil['means'][0], r_soil['sds'][0]),
     fmt(r_soil['means'][1], r_soil['sds'][1]), fmt(r_soil['means'][2], r_soil['sds'][2]),
     f"ANOVA p={r_soil['p']:.3f}", f"not detected; power={r_soil['power']:.2f}"],
    ['Subsoil LF-C 20-40 cm (g/m2)', f'{ref.mean():.0f}', f'{dlc.mean():.0f}',
     f'{np.nanmean(sts):.0f} (3/6 censored)', f'ANOVA p={lf_p_range} (censoring sensitivity)',
     'DLC/REF = 2.2x; Welch t p=0.007; robust to all substitution choices'],
    ['Ecosystem C total (Mg/ha)', f"{eco.loc['REF', 'Total']:.0f}", f"{eco.loc['DLC', 'Total']:.0f}",
     f"{eco.loc['STS', 'Total']:.0f}", 'descriptive (n=1 watershed/treatment)',
     f"% belowground: {eco.loc['REF', '% belowground']:.0f} / {eco.loc['DLC', '% belowground']:.0f} / {eco.loc['STS', '% belowground']:.0f}"],
], columns=['Variable', 'REF', 'DLC', 'STS', 'Primary test', 'Notes'])
table1


## Section 6 — Limitations & open items (the honest list, for Methods/Discussion)

1. **Pseudoreplication / scope of inference.** One watershed per treatment; plots are the replication unit within
   watershed. Inference is conditional on WS2/WS4/WS5, not on "diameter-limit cutting" as an abstract treatment
   (Hurlbert 1984, 2004; Oksanen 2001) — the manuscript states this in Methods, not as a buried caveat.
2. **Time since harvest (corrected from written records).** DLC plots ~2 yr, STS plots ~3.5 yr post-harvest (one
   plot each in the longer-rotation B subcompartments: 5 yr and ~9.5 yr). The earlier "2 vs 9 yr" confound came from
   an unverified 2013 date and no longer applies; DLC vs. STS is framed as an intensity contrast (notebook 02).
3. **Forest-floor (OH) stocks are under QC review** — recorded OH dry weights imply ~1.5 Mg/ha of forest floor, ~35×
   below measured WV FIA duff+litter. OH concentrations and C:N are valid; OH stocks (and the OH bar in Fig. 3) should
   not be interpreted until the dry weights are confirmed as whole-frame masses (notebook 03, QC-2).
4. **Aboveground–belowground pairing is watershed-level** — soil plots carry subcompartment, not plot, identity.
5. **Storage ≠ sequestration.** Only 2022 stocks are quantified; no rate is computed (no verified time series).
6. **No usable soil pH** for the study watersheds (the only pH file is from the unrelated WS3/WS7 acidification
   experiment) — stated as a limitation + future-work line, given pH's control on MAOM formation.
7. **Fraction recovery** 86–113% of bulk C — disclosed; figure basis controlled by `SCALE_FRACTIONS_TO_BULK`.


---
*All statistics computed at runtime from `trees.csv`, `soil_samples.csv`, `soil_fractions.csv` (folder set in the
config cell). Helpers are pure scipy/numpy so the notebook runs identically in ArcGIS Pro's default environment
and plain Jupyter. Dunnett's test uses scipy ≥ 1.11 when present and a labeled Holm-corrected Welch fallback
otherwise, so results are never silently different between environments.*
